In [1]:
!pip install numpy scipy casadi pyserial matplotlib pyserial
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
"""
validar_replay.py — Validación de métricas y re-play fiel al lazo en vivo
=========================================================================
Usa EXACTAMENTE el código de los notebooks (lazocerrado.ipynb y kalman.ipynb):
  - mismo modelo, mismo EDMD (semilla 42), mismo diseño de L (lazo alpha=1,10,...),
  - mismo EKF (Jacobiano numérico, clip 0.75, P0/Q/R),
  - mismo MPC (IPOPT, N=20, warm-start desplazado).

Qué hace:
  0. Imprime rho(A-LC), |L|max y tau_e del observador REAL (para compararlos con la Sec. 3.2.1 del artículo).
  1. Reproduce cada corrida con su propio observador y compara con el log y con la Tabla 4,
     usando las DOS ventanas (LIVE: t>400 s; PAPER: todo menos 60 s tras cada escalón).
  2. Cross-replay: ambos observadores sobre las 10 corridas (mismas u y medidas).
  3. Métricas por segmento y sesgo físico (GT - ref).
  4. Tiempos: Jacobiano (numérico vs CasADi) y solver del MPC con z0 nativo vs psi(x_EKF).

Uso:
  python validar_replay.py --csv-dir C:/Users/PC/Downloads/articulo
  python validar_replay.py --csv-dir . --no-mpc          # sin perfilar el MPC
Requiere: numpy scipy pandas casadi
"""
import numpy as np
import scipy.linalg as la
from scipy.optimize import least_squares
import casadi as ca
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# PARÁMETROS FÍSICOS — CALIBRADOS DESDE MEDICIÓN REAL
# ============================================================================
# Calibración de alphao (Torricelli):
#   Vaciado tanque 1: h=0.15m → h=0.05m en t=105s, bombas apagadas
#   Integración analítica:
#     alphao * Ao = Atank * 2*(sqrt(0.15)-sqrt(0.05)) / (sqrt(2*g) * 105)
#     alphao * Ao = 3.02e-3 * 0.3274 / (4.429 * 105) = 2.13e-6 m^2.5/s
#   Con Do=10mm → Ao = pi*(0.01)^2/4 = 7.85e-5 m²
#     alphao = 2.13e-6 / 7.85e-5 = 0.0271
#   Se asume mismo alphao para T1, T2, T3 (mismos orificios de fondo)
#
# Conexiones inter-tanques: Do=10mm (medido), A=7.85e-5 m²
#   alpha120/alpha230 se conservan del ajuste original (flujo viscoso)
#
P_NOM = {
    # --- Geometría de los tanques (cilíndricos, ø=6.2 cm) ---
    'Atank'   : 3.02e-3,     # m²  — π*(0.062/2)²

    # --- Fluidos ---
    'rho'     : 997.0,       # kg/m³
    'eta'     : 8.9e-4,      # Pa·s
    'g'       : 9.81,        # m/s²

    # --- Orificios de salida — CALIBRADOS desde vaciado 15→5 cm en 105 s ---
    #     alphao = 0.0271, Do = 10 mm → Ao = 7.85e-5 m²
    'alphao1' : 0.0271, 'Do1': 10e-3,   # orificio tanque 1
    'alphao2' : 0.0271, 'A2' : 7.85e-5, # orificio tanque 2 (mismo tipo)
    'alphao3' : 0.0271, 'Do3': 10e-3,   # orificio tanque 3

    # --- Conexiones inter-tanques — Do=10mm medido ---
    'alpha120': 0.3038, 'D12': 10e-3, 'A12': 7.85e-5, 'lambdac12': 24000,
    'alpha230': 0.1344, 'D23': 10e-3, 'A23': 7.85e-5, 'lambdac23': 29600,

    # --- Caudales máximos bombas (ya limitadas al 70%) ---
    'qi1max'  : 2.69e-5,     # m³/s — bomba 1 al 100% de señal → 70% real
    'qi3max'  : 2.41e-5,     # m³/s — bomba 3 al 100% de señal → 70% real
}

# ============================================================================
# PARÁMETROS DE SIMULACIÓN / CONTROL
# ============================================================================
Ts        = 2.0        # s — periodo de muestreo (igual que ESP32)
T_sim     = 800        # s — duración total del experimento
Nsim      = int(T_sim / Ts)
sigma     = 1e-3       # m — ruido de medición (1 mm, ajusta si es ruidoso)
EPS       = 1e-10
EPS_PSI   = 1e-8
SETTLING_S = 60        # s — ventana de transitorio excluida en métricas

# Puerto serial
SERIAL_PORT = 'COM6'
BAUD_RATE   = 115200
SERIAL_TIMEOUT = 3.0   # s

# Secuencia de referencias (en metros)
# 0.10 → 0.20 → 0.15 m  (la bajada 0.20→0.15 está calibrada para que las bombas aguanten)
REF_STEPS = [
    (0,           0.10),   # 0..266s   → h2 = 10 cm
    (int(Nsim*0.33), 0.20),  # 267..532s → h2 = 20 cm
    (int(Nsim*0.66), 0.15),  # 533..800s → h2 = 15 cm
]

# ============================================================================
# MODELO INTERNO (para EDMD y predictor Koopman)
# ============================================================================
def nl_ode(x, u, p=P_NOM):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = p['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h1)
    qo2 = p['alphao2'] * p['A2']                   * np.sqrt(2 * p['g'] * h2)
    qo3 = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h3)
    dh12 = h1 - h2
    l12  = p['D12'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS)
    a12  = p['alpha120'] * np.tanh(2 * l12 / p['lambdac12'])
    q12  = a12 * p['A12'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS) * np.sign(dh12 + 1e-15)
    dh23 = h2 - h3
    l23  = p['D23'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS)
    a23  = p['alpha230'] * np.tanh(2 * l23 / p['lambdac23'])
    q23  = a23 * p['A23'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS) * np.sign(dh23 + 1e-15)
    return np.array([
        (qi1 - q12 - qo1) / p['Atank'],
        (q12 - q23 - qo2) / p['Atank'],
        (qi3 + q23 - qo3) / p['Atank'],
    ])

def rk4(x, u, p=P_NOM, dt=Ts):
    k1 = nl_ode(x, u, p); k2 = nl_ode(x + dt/2*k1, u, p)
    k3 = nl_ode(x + dt/2*k2, u, p); k4 = nl_ode(x + dt*k3, u, p)
    return np.clip(x + dt/6*(k1 + 2*k2 + 2*k3 + k4), 0, 0.27)  # antes 0.25

def psi(x):
    """8 observables de Koopman."""
    h1, h2, h3 = x[0], x[1], x[2]
    return np.array([
        np.sqrt(max(h1, EPS_PSI)),
        np.sqrt(max(h2, EPS_PSI)),
        np.sqrt(max(h3, EPS_PSI)),
        h1 * h2, h2 * h3, h1 * h3,
        h1, h3,
    ], dtype=float)

def compute_ss(h2_ref, p=P_NOM):
    """
    Punto de operación estacionario para h2=h2_ref.
    Permite u3>0 (regularizado para preferir u3≈0 cuando sea posible),
    ya que para h2=0.20m puede no existir SS factible con u3=0 dentro
    del rango físico [0,0.25]m.
    """
    w_reg = 0.05  # penaliza u3 para preferir soluciones con u3≈0 si existen
    def eqs(v):
        h1, h3, u1, u3 = v
        f = nl_ode([max(h1, 1e-4), h2_ref, max(h3, 1e-4)],
                   [np.clip(u1, 0, 1), np.clip(u3, 0, 1)], p)
        return np.concatenate([f, [w_reg * u3]])

    guesses = [
        (h2_ref*1.4, h2_ref*0.8, 0.5, 0.0),
        (h2_ref*1.6, h2_ref*0.7, 0.7, 0.0),
        (h2_ref*2.0, h2_ref*0.8, 0.9, 0.0),
        (h2_ref*2.5, h2_ref*0.8, 0.7, 0.0),
        (h2_ref*2.5, h2_ref*0.8, 0.75, 0.0),
        (h2_ref*1.8, h2_ref*0.85, 0.85, 0.0),
        (h2_ref*1.3, h2_ref*0.9, 1.0, 0.1),
        (h2_ref*1.2, h2_ref*1.0, 1.0, 0.3),
        (h2_ref*1.1, h2_ref*1.1, 1.0, 0.5),
        (h2_ref*1.0, h2_ref*1.2, 1.0, 0.8),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01, 0.01, 0.0, 0.0],
                                      [0.27, 0.27, 1.0, 1.0]),
                              max_nfev=5000)
            phys_res = np.linalg.norm(s.fun[:3])  # solo las 3 ecuaciones físicas
            if s.success and phys_res < best_res:
                best_res = phys_res; best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return (np.array([best[0], h2_ref, best[1]]),
                np.array([np.clip(best[2], 0, 1), np.clip(best[3], 0, 1)]))
    return None, None


# ============================================================================
# ENTRENAMIENTO EDMD
# ============================================================================
def train_edmd(verbose=True):
    if verbose:
        print("\nEntrenando EDMD (parámetros nominales)...")

    rng = np.random.default_rng(42)
    z_k, u_k, z_k1 = [], [], []

    # Bloque 1: perturbaciones normales alrededor de los SS
    for h2r in [0.10, 0.12, 0.15, 0.18, 0.20, 0.22]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None:
            continue
        for pert in [0.95, 0.90, 0.98, 0.85, 0.92, 0.88]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.05, 0.70)
            for _ in range(170):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.35*err, -0.35, 0.35) + rng.uniform(-0.15, 0.15)
                u[1] += rng.uniform(-0.05, 0.05); u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    # Bloque 2: perturbaciones grandes ±40%
    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None:
            continue
        for pert in [0.60, 0.70, 0.75, 1.25, 1.35]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.05, 0.70)
            for _ in range(100):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.6*err, -0.6, 0.6) + rng.uniform(-0.1, 0.1)
                u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    # Bloque 3: tránsitos entre referencias
    for (h2f, h2t) in [(0.10, 0.20), (0.20, 0.15), (0.15, 0.10)]:
        xf, uf = compute_ss(h2f); xt, _ = compute_ss(h2t)
        if xf is None or xt is None:
            continue
        x = xf.copy()
        for _ in range(200):
            err = x[1] - h2t; u = uf.copy()
            u[0] += np.clip(-0.5*err, -0.5, 0.5) + rng.uniform(-0.08, 0.08)
            u = np.clip(u, 0, 1)
            z = psi(x); xn = rk4(x, u); zn = psi(xn)
            z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    Z = np.array(z_k); U = np.array(u_k); Zn = np.array(z_k1)
    nz, nu = Z.shape[1], U.shape[1]
    if verbose:
        print(f"  {len(Z)} muestras de entrenamiento")

    # Regresión EDMD con Tikhonov
    Phi = np.hstack([Z, U]); lam = 1e-4
    Theta = np.linalg.solve(Phi.T @ Phi + lam * np.eye(nz + nu), Phi.T @ Zn)
    A_raw = Theta[:nz, :].T; B_d = Theta[nz:, :].T

    # Estabilización espectral
    vals, vecs = np.linalg.eig(A_raw); delta = 1e-4
    vals_s = np.where(np.abs(vals) >= 1.0, vals / np.abs(vals) * (1.0 - delta), vals)
    A_d = np.real(vecs @ np.diag(vals_s) @ np.linalg.inv(vecs))

    rmse = np.sqrt(np.mean((Z @ A_d.T + U @ B_d.T - Zn)**2))
    eigs = np.abs(np.linalg.eigvals(A_d))
    if verbose:
        print(f"  RMSE one-step: {rmse:.2e}")
        print(f"  |λ| ∈ [{eigs.min():.5f}, {eigs.max():.5f}] ✓")
    return A_d, B_d, nz, nu

def design_observer(A_d, nz):
    """Diseña ganancia Luenberger por DARE."""
    # C_obs mide √h1, √h3, h1, h3 (filas 0,2,6,7 del vector observable)
    C_obs = np.zeros((4, nz))
    C_obs[0, 0] = 1.0   # √h1
    C_obs[1, 2] = 1.0   # √h3
    C_obs[2, 6] = 1.0   # h1
    C_obs[3, 7] = 1.0   # h3
    m = C_obs.shape[0]

    print("\nDiseñando ganancia L (Luenberger)...")
    for alpha in [1.0, 10.0, 50.0, 100.0, 500.0, 1000.0]:
        try:
            Q_obs = alpha * np.eye(nz)
            X = la.solve_discrete_are(A_d.T, C_obs.T, Q_obs, np.eye(m))
            L_c = np.linalg.solve(C_obs @ X @ C_obs.T + np.eye(m), C_obs @ X @ A_d.T).T
            eigs_L = np.abs(np.linalg.eigvals(A_d - L_c @ C_obs)).max()
            if eigs_L < 0.999 and np.abs(L_c).max() < 500:
                print(f"  alpha={alpha}: max|λ|={eigs_L:.5f} ✓")
                return L_c, C_obs
        except:
            pass
    raise RuntimeError("No se pudo diseñar L. Revisa los parámetros del modelo.")

def build_mpc(A_d, B_d, nz, nu):
    """Construye el optimizador KoopmanMPC con CasADi/IPOPT."""
    print("\nConstruyendo KoopmanMPC...")
    N_mpc = 20; q_mpc = 500
    R1 = np.diag([0.1, 1.0]); R2 = np.diag([0.1, 0.1])
    umin = np.array([0.0, 0.0]); umax = np.array([1.0, 1.0])

    Z_sym = ca.SX.sym('Z', nz, N_mpc+1)
    U_sym = ca.SX.sym('U', nu, N_mpc)
    z0_sym = ca.SX.sym('z0', nz)
    yr_sym = ca.SX.sym('yr')
    up_sym = ca.SX.sym('up', nu)

    A_cas = ca.DM(A_d); B_cas = ca.DM(B_d)
    J = 0; g = []; lbg = []; ubg = []

    g += [Z_sym[:, 0] - z0_sym]; lbg += [0.0]*nz; ubg += [0.0]*nz
    for k in range(N_mpc):
        g += [Z_sym[:, k+1] - A_cas@Z_sym[:, k] - B_cas@U_sym[:, k]]
        lbg += [0.0]*nz; ubg += [0.0]*nz
        h2_k = Z_sym[1, k+1]**2; ey = h2_k - yr_sym
        du = U_sym[:, 0] - up_sym if k == 0 else U_sym[:, k] - U_sym[:, k-1]
        wk = 10 if k == N_mpc-1 else 1
        J += (wk * q_mpc * ey**2
              + ca.mtimes(U_sym[:, k].T, ca.DM(R1) @ U_sym[:, k])
              + ca.mtimes(du.T,          ca.DM(R2) @ du))

    w_sym  = ca.vertcat(ca.reshape(Z_sym, nz*(N_mpc+1), 1), ca.reshape(U_sym, nu*N_mpc, 1))
    par_sym = ca.vertcat(z0_sym, yr_sym, up_sym)
    lbw = ([-ca.inf]*(nz*(N_mpc+1)) + list(np.tile(umin, N_mpc)))
    ubw = ([ ca.inf]*(nz*(N_mpc+1)) + list(np.tile(umax, N_mpc)))

    nlp = {'x': w_sym, 'f': J, 'g': ca.vertcat(*g), 'p': par_sym}
    opts = {'ipopt.print_level': 0, 'ipopt.max_iter': 500,
            'ipopt.tol': 1e-5, 'ipopt.acceptable_tol': 1e-4, 'print_time': 0}
    solver = ca.nlpsol('solver_kmpc', 'ipopt', nlp, opts)
    print("  KoopmanMPC ✓")
    return solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz, nu


# ============================================================================
# PARÁMETROS EKF
# ============================================================================
# Covarianza ruido de proceso Q (3x3) — cuánto confías en el modelo
# Diagonal: [varianza_h1, varianza_h2, varianza_h3] en m²
# h2 tiene mayor incertidumbre porque no se mide directamente
EKF_Q = np.diag([
    (0.5e-3)**2,   # h1: ruido de proceso ~0.5 mm
    (2.0e-3)**2,   # h2: mayor incertidumbre (no medido) ~2 mm
    (0.5e-3)**2,   # h3: ruido de proceso ~0.5 mm
])

# Covarianza ruido de medición R (2x2) — ruido del ultrasónico
# HC-SR04 tiene ruido típico de ~1-2 mm
EKF_R = np.diag([
    (1.5e-3)**2,   # h1 medido: ~1.5 mm de ruido
    (1.5e-3)**2,   # h3 medido: ~1.5 mm de ruido
])

# Covarianza inicial P0 — incertidumbre inicial del estado
EKF_P0 = np.diag([
    (5e-3)**2,     # h1: incertidumbre inicial ~5 mm
    (20e-3)**2,    # h2: alta incertidumbre inicial (no medido) ~20 mm
    (5e-3)**2,     # h3: incertidumbre inicial ~5 mm
])

# Filtro anti-outlier del ultrasónico (igual que Koopman)
MAX_DELTA_H = 0.05   # m — salto máximo físicamente posible en 2s


# ============================================================================
class EKF:
    """
    Extended Kalman Filter para el sistema de tres tanques.

    Estado: x = [h1, h2, h3]  (metros)
    Medición: y = [h1, h3]     (metros, solo tanques 1 y 3)
    h2 se estima sin medirlo directamente.

    Ciclo por paso de muestreo:
      1. predict(u)  — integra RK4 + propaga covarianza con Jacobiano numérico
      2. update(y)   — corrección bayesiana con medición [h1, h3]
    """

    # Matriz de observación: mide h1 (fila 0) y h3 (fila 2)
    C = np.array([[1.0, 0.0, 0.0],
                  [0.0, 0.0, 1.0]])

    def __init__(self, x0, P0=None, Q=None, R=None, p=P_NOM):
        self.x = np.array(x0, dtype=float)
        self.P = P0.copy() if P0 is not None else EKF_P0.copy()
        self.Q = Q.copy()  if Q  is not None else EKF_Q.copy()
        self.R = R.copy()  if R  is not None else EKF_R.copy()
        self.p = p

    def _jacobian(self, x, u, eps=1e-6):
        """
        Jacobiano numérico Fk = ∂f/∂x evaluado en (x, u).
        Se discretiza como Fk ≈ I + Ts * J_continuo.
        """
        n = len(x)
        J = np.zeros((n, n))
        f0 = nl_ode(x, u, self.p)
        for i in range(n):
            xp = x.copy(); xp[i] += eps
            J[:, i] = (nl_ode(xp, u, self.p) - f0) / eps
        # Jacobiano discreto de primer orden
        return np.eye(n) + Ts * J

    def predict(self, u):
        """
        Etapa de predicción:
          x̂⁻ = RK4(x̂, u)
          P⁻  = Fk @ P @ Fk.T + Q
        """
        Fk      = self._jacobian(self.x, u)
        self.x  = rk4(self.x, u, self.p, Ts)
        self.x  = np.clip(self.x, 0.0, 0.75)
        self.P  = Fk @ self.P @ Fk.T + self.Q

    def update(self, y_meas):
        """
        Etapa de corrección con y = [h1_medido, h3_medido]:
          S  = C @ P⁻ @ C.T + R
          K  = P⁻ @ C.T @ S⁻¹
          x̂  = x̂⁻ + K @ (y - C @ x̂⁻)
          P  = (I - K @ C) @ P⁻
        Devuelve la estimación corregida.
        """
        y    = np.array(y_meas, dtype=float)
        S    = self.C @ self.P @ self.C.T + self.R
        K    = self.P @ self.C.T @ np.linalg.inv(S)
        innov = y - self.C @ self.x
        self.x = self.x + K @ innov
        self.x = np.clip(self.x, 0.0, 0.75)
        self.P = (np.eye(3) - K @ self.C) @ self.P
        # Simetrizar para evitar deriva numérica
        self.P = 0.5 * (self.P + self.P.T)
        return self.x.copy()

    @property
    def h2_estimate(self):
        return float(self.x[1])

    @property
    def std_h2(self):
        """Desviación estándar de la estimación de h2 (raíz de P[1,1])."""
        return float(np.sqrt(max(self.P[1, 1], 0.0)))



# ============================================================================
# >>> A PARTIR DE AQUÍ: CÓDIGO NUEVO DE VALIDACIÓN (replay fiel al lazo en vivo)
# ============================================================================
import argparse, glob, os, re, time, sys
import pandas as pd

# Carpeta con los CSV (editar si hace falta). En Jupyter se usa este valor.
CSV_DIR_DEFAULT = r"C:/Users/PC/Downloads/articulo" if os.path.isdir(r"C:/Users/PC/Downloads/articulo") else "."

# Valores publicados (Tabla 4 del artículo) para comprobar la reproducción
TABLA4 = {
    'koopman': {'track': [0.575, 0.615, 0.637, 0.577, 0.650], 'obs': [0.907, 0.831, 0.986, 0.839, 0.925]},
    'ekf':     {'track': [0.760, 0.885, 0.920, 0.819, 0.811], 'obs': [0.984, 1.056, 1.145, 1.032, 1.052]},
}

def rmse_cm(e):
    return float(np.sqrt(np.mean(np.square(e)))) if len(e) else float('nan')

# ---- Ventanas de steady-state ------------------------------------------------
def mask_live(T):
    """Ventana del código en vivo (compute_metrics): t>400 y sin los 60 s tras cada cambio."""
    m = (T > 400)
    for tc in (264.0, 528.0):
        m &= ~((T >= tc) & (T < tc + SETTLING_S))
    return m

def mask_paper(T):
    """Ventana descrita en la Sec. 3.5 del artículo: todo el registro menos 60 s tras CADA escalón (incl. t=0)."""
    m = np.ones(len(T), dtype=bool)
    for tc in (0.0, 264.0, 528.0):
        m &= ~((T >= tc) & (T < tc + SETTLING_S))
    return m

SEGS = [(0, 264, 0.10), (264, 528, 0.20), (528, 800, 0.15)]
def seg_masks(T):
    return [(T >= t0 + SETTLING_S) & (T < t1) for t0, t1, _ in SEGS]

# ---- Filtro anti-outlier idéntico al del EKF en vivo -----------------------------
def outlier_filter(x, max_delta=MAX_DELTA_H):
    y = np.array(x, dtype=float)
    for k in range(1, len(y)):
        if abs(y[k] - y[k-1]) > max_delta:
            y[k] = y[k-1]
    return y

# ---- Carga de CSV -------------------------------------------------------------
def load_log(path):
    df = pd.read_csv(path); df.columns = df.columns.str.strip()
    est = next(c for c in ('h2_est_m', 'h2_ekf_m') if c in df.columns)
    d = dict(T=df['t_s'].values, h1=df['h1_meas_m'].values, h3=df['h3_meas_m'].values,
             h2r=df['h2_real_m'].values, ref=df['ref_m'].values,
             u=np.column_stack([df['u1'].values, df['u3'].values]),
             h2_log=df[est].values)
    return d

# ---- Replays fieles al lazo en vivo ------------------------------------------------
def replay_koopman(d, A, B, L, C, h1=None, h3=None):
    h1 = d['h1'] if h1 is None else h1
    h3 = d['h3'] if h3 is None else h3
    n = len(d['T'])
    z = psi(np.array([h1[0], d['ref'][0], h3[0]]))
    uprev = np.zeros(2)
    X = np.zeros((n, 3)); Z = np.zeros((n, A.shape[0]))
    X[0] = [h1[0], d['ref'][0], h3[0]]; Z[0] = z
    for k in range(n - 1):
        zp = A @ z + B @ uprev
        zp[:3] = np.maximum(zp[:3], 0); zp[6:] = np.maximum(zp[6:], 0)
        y = np.array([np.sqrt(max(h1[k+1], EPS_PSI)), np.sqrt(max(h3[k+1], EPS_PSI)), h1[k+1], h3[k+1]])
        z = zp + L @ (y - C @ zp)
        z[:3] = np.maximum(z[:3], 0); z[6:] = np.maximum(z[6:], 0)
        X[k+1] = np.clip(z[:3]**2, 0, 0.75); Z[k+1] = z
        uprev = d['u'][k].copy()
    return X, Z

def replay_ekf(d, h1=None, h3=None):
    h1 = d['h1'] if h1 is None else h1
    h3 = d['h3'] if h3 is None else h3
    n = len(d['T'])
    ekf = EKF(x0=np.array([h1[0], d['ref'][0], h3[0]]), P0=EKF_P0, Q=EKF_Q, R=EKF_R)
    uprev = np.zeros(2)
    X = np.zeros((n, 3)); X[0] = ekf.x
    for k in range(n - 1):
        ekf.predict(uprev)
        X[k+1] = ekf.update(np.array([h1[k+1], h3[k+1]]))
        uprev = d['u'][k].copy()
    return X

# ---- Métricas ----------------------------------------------------------------------
def metrics(d, h2_est, which):
    T = d['T']; m = mask_live(T) if which == 'live' else mask_paper(T)
    e_obs = (h2_est - d['h2r']) * 100
    e_trk = (h2_est - d['ref']) * 100
    return rmse_cm(e_trk[m]), rmse_cm(e_obs[m]), float(np.mean(e_trk[m])), float(np.mean(e_obs[m]))

def seg_table(d, h2_est):
    T = d['T']; out = []
    for m in seg_masks(T):
        out.append((rmse_cm(((h2_est - d['ref']) * 100)[m]), rmse_cm(((h2_est - d['h2r']) * 100)[m]),
                    float(np.mean(((h2_est - d['ref']) * 100)[m])), float(np.mean(((h2_est - d['h2r']) * 100)[m]))))
    return out

def run_num(p):
    m = re.search(r'run(\d+)', os.path.basename(p)); return int(m.group(1)) if m else -1

def find_logs(csv_dir, prefix):
    return sorted(glob.glob(os.path.join(csv_dir, f'{prefix}_run*.csv')), key=run_num)

# ---- Perfilado del MPC (mismo solver IPOPT y warm-start desplazado que en vivo) -----------------
def profile_mpc(solver_pack, z_seq, d, label):
    solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz, nu = solver_pack
    n = len(d['T']); w0 = None; yr_prev = d['ref'][0]
    t_ms, iters, fails = [], [], 0
    uprev = np.zeros(2)
    for k in range(1, n):
        yr = d['ref'][k-1]            # en vivo: yr = yref_v[k] en el paso que genera la fila k
        if yr != yr_prev: w0 = None
        yr_prev = yr
        z0 = z_seq[k]
        if w0 is None:
            w0 = np.zeros(nz*(N_mpc+1) + nu*N_mpc); w0[:nz] = z0
        par = np.concatenate([z0, [yr], uprev])
        t0 = time.perf_counter()
        try:
            sol = solver(x0=w0, lbx=lbw, ubx=ubw, lbg=lbg, ubg=ubg, p=par)
            t1 = time.perf_counter()
            w = np.array(sol['x']).flatten()
            Zo = w[:nz*(N_mpc+1)].reshape(N_mpc+1, nz); Uo = w[nz*(N_mpc+1):].reshape(N_mpc, nu)
            w0 = np.concatenate([np.vstack([Zo[1:], Zo[-1:]]).flatten(), np.vstack([Uo[1:], Uo[-1:]]).flatten()])
            t_ms.append((t1 - t0) * 1000); iters.append(solver.stats().get('iter_count', np.nan))
        except Exception:
            fails += 1; w0 = None
        uprev = d['u'][k-1].copy()    # el u aplicado es el LOGGEADO (así la secuencia z es consistente)
    t_ms = np.array(t_ms); iters = np.array(iters, dtype=float)
    print(f"   {label:<34} solver: media {t_ms.mean():7.2f} ms | P99 {np.percentile(t_ms,99):7.2f} ms | "
          f"iter media {np.nanmean(iters):5.1f} | fallos {fails}")
    return t_ms, iters

def time_jacobians(x=np.array([0.15, 0.12, 0.10]), u=np.array([0.6, 0.1]), reps=2000):
    ekf = EKF(x0=x, P0=EKF_P0, Q=EKF_Q, R=EKF_R)
    t0 = time.perf_counter()
    for _ in range(reps): ekf._jacobian(x, u)
    t_num = (time.perf_counter() - t0) / reps * 1000
    t_ad = None
    try:
        import casadi as ca
        xs = ca.SX.sym('x', 3); us = ca.SX.sym('u', 2)
        # Jacobiano algorítmico del campo ODE (mismo modelo, CasADi)
        h = [ca.fmax(xs[i], EPS) for i in range(3)]
        p = P_NOM
        qi1 = p['qi1max']*ca.fmin(ca.fmax(us[0], 0), 1); qi3 = p['qi3max']*ca.fmin(ca.fmax(us[1], 0), 1)
        qo1 = p['alphao1']*(p['Do1']**2*np.pi/4)*ca.sqrt(2*p['g']*h[0])
        qo2 = p['alphao2']*p['A2']*ca.sqrt(2*p['g']*h[1])
        qo3 = p['alphao3']*(p['Do3']**2*np.pi/4)*ca.sqrt(2*p['g']*h[2])
        def cpl(da, al0, D, A, lc):
            s = ca.sqrt(2*p['g']*ca.fabs(da) + EPS)
            return p[al0]*ca.tanh(2*(p[D]*p['rho']/p['eta']*s)/p[lc])*p[A]*s*ca.sign(da + 1e-15)
        q12 = cpl(h[0]-h[1], 'alpha120', 'D12', 'A12', 'lambdac12')
        q23 = cpl(h[1]-h[2], 'alpha230', 'D23', 'A23', 'lambdac23')
        f = ca.vertcat((qi1-q12-qo1)/p['Atank'], (q12-q23-qo2)/p['Atank'], (qi3+q23-qo3)/p['Atank'])
        Fk = ca.Function('Fk', [xs, us], [ca.SX.eye(3) + Ts*ca.jacobian(f, xs)])
        Fk(x, u)
        t0 = time.perf_counter()
        for _ in range(reps): Fk(x, u)
        t_ad = (time.perf_counter() - t0) / reps * 1000
    except Exception as e:
        print("   (CasADi no disponible para el Jacobiano algorítmico:", e, ")")
    return t_num, t_ad

# =============================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--csv-dir', default=CSV_DIR_DEFAULT)
    ap.add_argument('--no-mpc', action='store_true', help='omitir el perfilado del MPC (IPOPT)')
    ap.add_argument('--mpc-runs', type=int, default=1, help='cuántas corridas perfilar (por método)')
    a, _ = ap.parse_known_args()   # ignora los argumentos que Jupyter agrega (--f=...)

    print("="*78); print(" 0. Observador de Koopman EXACTO del código en vivo"); print("="*78)
    A, B, nz, nu = train_edmd(verbose=True)
    L, C = design_observer(A, nz)
    rho = float(np.max(np.abs(np.linalg.eigvals(A - L @ C))))
    print(f"   rho(A)={np.max(np.abs(np.linalg.eigvals(A))):.5f} | rho(A-LC)={rho:.5f} | |L|max={np.abs(L).max():.4f} | "
          f"tau_e = -Ts/ln(rho) = {-Ts/np.log(rho):.1f} s")
    print("   (El artículo, Sec. 3.2.1 y Remark 1, dice rho=0.99847, |L|max=0.001, tau=1306 s: COMPARA.)")

    kfiles = find_logs(a.csv_dir, 'koopman'); efiles = find_logs(a.csv_dir, 'ekf')
    if not kfiles or not efiles:
        sys.exit(f"No encontré koopman_run*.csv / ekf_run*.csv en {a.csv_dir}")
    K = [load_log(p) for p in kfiles]; E = [load_log(p) for p in efiles]

    # ---------------------------------------------------------------- 1. reproducción
    print("\n"+"="*78); print(" 1. REPRODUCCIÓN: cada observador sobre SU PROPIA corrida (debe igualar el log y la Tabla 4)")
    print("="*78)
    print("   max|dif| = máx |h2_replay - h2_log| [cm]. Debe ser ~0. Ventanas: LIVE = t>400 s; PAPER = todo menos 60 s tras cada escalón.")
    print(f"   {'':<10}{'maxdif':>8} | {'obs LIVE':>9}{'obs PAPER':>10}{'Tab.4':>8} | {'trk LIVE':>9}{'trk PAPER':>10}{'Tab.4':>8}")
    for name, runs, files in (('koopman', K, kfiles), ('ekf', E, efiles)):
        for i, (d, f) in enumerate(zip(runs, files)):
            X = replay_koopman(d, A, B, L, C)[0] if name == 'koopman' else replay_ekf(d)
            md = np.max(np.abs(X[:, 1] - d['h2_log'])) * 100
            tl = metrics(d, X[:, 1], 'live'); tp = metrics(d, X[:, 1], 'paper')
            ref = TABLA4[name]; j = min(i, 4)
            print(f"   {name[:4]}{run_num(f):>3}   {md:8.3f} | {tl[1]:9.3f}{tp[1]:10.3f}{ref['obs'][j]:8.3f} | "
                  f"{tl[0]:9.3f}{tp[0]:10.3f}{ref['track'][j]:8.3f}")

    # ---------------------------------------------------------------- 2. cross-replay
    print("\n"+"="*78); print(" 2. CROSS-REPLAY: cada observador sobre las 10 corridas (mismas u y medidas, filtro anti-outlier para ambos)")
    print("="*78)
    print("   RMSE de observación (est vs h2 real) [cm], ventana LIVE / PAPER")
    cross = {}
    for src_name, runs, files in (('log Koopman', K, kfiles), ('log EKF', E, efiles)):
        ko_l, ko_p, ek_l, ek_p = [], [], [], []
        for d in runs:
            h1f = outlier_filter(d['h1']); h3f = outlier_filter(d['h3'])
            Xk = replay_koopman(d, A, B, L, C, h1f, h3f)[0][:, 1]; Xe = replay_ekf(d, h1f, h3f)[:, 1]
            ko_l.append(metrics(d, Xk, 'live')[1]); ko_p.append(metrics(d, Xk, 'paper')[1])
            ek_l.append(metrics(d, Xe, 'live')[1]); ek_p.append(metrics(d, Xe, 'paper')[1])
        cross[src_name] = (ko_l, ko_p, ek_l, ek_p)
        print(f"   Sobre {src_name:<12}  Koopman: LIVE {np.mean(ko_l):.3f}±{np.std(ko_l,ddof=1):.3f} | PAPER {np.mean(ko_p):.3f}±{np.std(ko_p,ddof=1):.3f}"
              f"   ||   EKF: LIVE {np.mean(ek_l):.3f}±{np.std(ek_l,ddof=1):.3f} | PAPER {np.mean(ek_p):.3f}±{np.std(ek_p,ddof=1):.3f}")
    allk = cross['log Koopman'][0] + cross['log EKF'][0]; alle = cross['log Koopman'][2] + cross['log EKF'][2]
    print(f"   Las 10 corridas juntas (LIVE): Koopman {np.mean(allk):.3f} cm | EKF {np.mean(alle):.3f} cm")
    print("   -> Este es el único experimento que compara los observadores con entradas/medidas idénticas.")

    # ---------------------------------------------------------------- 3. por segmento y sesgo físico
    print("\n"+"="*78); print(" 3. POR SEGMENTO (60 s excluidos por tramo) y sesgo físico GT-ref = sesgo_trk - sesgo_obs  [cm]")
    print("="*78)
    for name, runs in (('koopman', K), ('ekf', E)):
        tabs = np.array([[s for s in seg_table(d, (replay_koopman(d, A, B, L, C)[0] if name == 'koopman' else replay_ekf(d))[:, 1])] for d in runs])
        print(f"   {name.upper()}  (media sobre {len(runs)} corridas)  [rmse_trk, rmse_obs, bias_trk, bias_obs]")
        for s, (t0, t1, r) in enumerate(SEGS):
            m = tabs[:, s, :].mean(axis=0)
            print(f"     Seg{s+1} ({r:.2f} m): trk {m[0]:.3f} | obs {m[1]:.3f} | bias_trk {m[2]:+.3f} | bias_obs {m[3]:+.3f} | sesgo físico GT-ref {m[2]-m[3]:+.3f}")

    # ---------------------------------------------------------------- 4. tiempos
    print("\n"+"="*78); print(" 4. TIEMPOS DE CÓMPUTO (solo cómputo, sin esperar al ESP32)")
    print("="*78)
    tn, ta = time_jacobians()
    print(f"   Jacobiano EKF por diferencias finitas (código en vivo): {tn:.4f} ms")
    if ta is not None: print(f"   Jacobiano algorítmico CasADi:                           {ta:.4f} ms")
    print("   (Nota: en el código en vivo step_ms incluye get_measurement(), que BLOQUEA hasta que el ESP32 envía el dato.")
    print("    Por eso los ~2100 ms de la Tabla 14 / Fig. 5 son tiempo de ciclo del ESP32, no cómputo del MPC ni del EKF.)")
    if not a.no_mpc:
        try:
            import casadi as ca
            pack = build_mpc(A, B, nz, nu)
            print("   Perfilado del MPC (IPOPT, max_iter=500, warm-start desplazado, como en vivo):")
            for i in range(min(a.mpc_runs, len(K))):
                dk = K[i]
                Zk = replay_koopman(dk, A, B, L, C)[1]                               # z nativo del observador de Koopman
                Xe = replay_ekf(dk)                                                    # EKF sobre el mismo log
                Ze = np.array([psi(x) for x in Xe])                                    # psi(x_hat_EKF)
                print(f"   -- log Koopman run {run_num(kfiles[i])}, mismas u y referencias:")
                profile_mpc(pack, Zk, dk, 'z0 nativo (observador Koopman)')
                profile_mpc(pack, Ze, dk, 'z0 = psi(x_hat EKF)')
        except Exception as e:
            print("   No se pudo perfilar el MPC:", e)
    print("\nListo.")

if __name__ == '__main__':
    main()

 0. Observador de Koopman EXACTO del código en vivo

Entrenando EDMD (parámetros nominales)...
  8220 muestras de entrenamiento
  RMSE one-step: 9.74e-05
  |λ| ∈ [0.03444, 0.99990] ✓

Diseñando ganancia L (Luenberger)...
  alpha=1.0: max|λ|=0.94262 ✓
   rho(A)=0.99990 | rho(A-LC)=0.94262 | |L|max=0.5867 | tau_e = -Ts/ln(rho) = 33.8 s
   (El artículo, Sec. 3.2.1 y Remark 1, dice rho=0.99847, |L|max=0.001, tau=1306 s: COMPARA.)

 1. REPRODUCCIÓN: cada observador sobre SU PROPIA corrida (debe igualar el log y la Tabla 4)
   max|dif| = máx |h2_replay - h2_log| [cm]. Debe ser ~0. Ventanas: LIVE = t>400 s; PAPER = todo menos 60 s tras cada escalón.
               maxdif |  obs LIVE obs PAPER   Tab.4 |  trk LIVE trk PAPER   Tab.4
   koop  1      0.000 |     0.907     1.093   0.907 |     0.575     0.568   0.575
   koop  2      0.000 |     0.831     1.106   0.831 |     0.615     0.598   0.615
   koop  3      0.000 |     0.986     1.231   0.986 |     0.637     0.610   0.637
   koop  4      0.000